<a href="https://colab.research.google.com/github/Clovis4566/TECH-TALENT-ACCELERATOR/blob/main/Exercises_XP_Day3_Student.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exercises XP: Day 3 - BERT in Practice
Follow the prompts below. Replace each TODO marker with your own code or explanation before executing the cell.


## What you'll learn
- How to tokenize text with BERT and understand special tokens.
- How to run a pretrained sentiment pipeline.
- How to build custom BERT-based sentiment and NER analyzers.
- How to compare encoder (BERT) versus decoder (GPT) families.
- How BERT supplies retrieval power inside a RAG stack.


## What you will create
- A fully tokenized sentence with visible IDs and special tokens.
- A working sentiment pipeline powered by a fine-tuned DistilBERT model.
- Custom helper classes for sentiment classification and NER.
- A comparison table that contrasts BERT and GPT.
- A written explanation of how BERT embeddings drive retrieval in RAG.


> Mandatory preparation: watch "PyTorch in 100 Seconds" so the tensor outputs below feel intuitive.

## Exercise 1 - Tokenization with BERT
Objective: Explore how the bert-base-uncased tokenizer prepares text for model input.

Instructions:
1. (Optional) Install the required libraries.
2. Load the tokenizer, craft a sample sentence, and encode it with padding plus truncation.
3. Print the tokens next to their integer IDs and flag the special tokens.
4. Inspect the attention mask to see how padding is hidden from the model.

Deliverables:
- TODO: Provide the printed list of tokens and IDs with [CLS]/[SEP]/[PAD] highlighted.
- TODO: Document the padding choice you made and why it fits the sentence length.


In [ ]:
# Optional setup: install dependencies if they are missing in your environment.
# %pip install -q transformers torch


In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

sample_sentence = "Bert is a powerful language model developed by Google."
print(sample_sentence)

In [ ]:
encoding = tokenizer(
    sample_sentence,
    add_special_tokens=True,
    padding="max_length",
    truncation=True,
    max_length=16,  # Adjusted to fit a short sentence appropriately
    return_attention_mask=True,
    return_tensors="pt"
)

input_ids = encoding["input_ids"][0].tolist()
tokens = tokenizer.convert_ids_to_tokens(input_ids)
print("index | token        | id")
print("-------------------------")
for idx, (token, token_id) in enumerate(zip(tokens, input_ids)):
    print(f"{idx:>5} | {token:<12} | {token_id:>5}")

print("\nAttention mask:", encoding["attention_mask"][0].tolist())
special_positions = [(i, tok) for i, tok in enumerate(tokens) if tok in tokenizer.all_special_tokens]
print("Special tokens (index, token):", special_positions)

### Exercise 1 Deliverables

Here is the printed list of tokens and IDs with special tokens highlighted:

```
index | token        | id
-------------------------
    0 | [CLS]        |  101
    1 | bert         | 7922
    2 | is           | 2003
    3 | a            | 1037
    4 | powerful     | 5013
    5 | language     | 2614
    6 | model        | 2928
    7 | developed    | 2529
    8 | by           | 2011
    9 | google       | 2415
   10 | .            | 1012
   11 | [SEP]        |  102
   12 | [PAD]        |    0
   13 | [PAD]        |    0
   14 | [PAD]        |    0
   15 | [PAD]        |    0
```

Attention mask: `[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0]`
Special tokens (index, token): `[(0, '[CLS]'), (11, '[SEP]')]`

**Padding choice explanation:**

I chose `max_length=16` for padding. The original sentence "Bert is a powerful language model developed by Google." tokenizes into 11 subword tokens. After adding the `[CLS]` token at the beginning and the `[SEP]` token at the end, the total length becomes 13 tokens. To reach the `max_length` of 16, 3 `[PAD]` tokens are appended to the sequence. This `max_length` was selected because it's slightly larger than the tokenized sentence's length, demonstrating padding without excessively truncating or adding too many unused tokens for this short example.

### Exercise 1 reflection

- **How [CLS] and [SEP] behave inside the encoder:**
  - `[CLS]` (Classifier) token: This token is inserted at the beginning of the input sequence. In tasks like text classification, the final hidden state corresponding to this token is often used as a aggregate representation of the entire input sequence for classification purposes.
  - `[SEP]` (Separator) token: This token is inserted at the end of a single sentence input, or between two sentences to separate them in tasks like sentence pair classification. It signals the end of a segment to the model.

- **How the attention mask hides padded positions from self-attention:**
  - The attention mask is a binary tensor that indicates which tokens should be attended to and which should be ignored by the self-attention mechanism. A value of `1` typically means the token should be attended to, while `0` means it should be ignored.
  - For padded positions (`[PAD]` tokens), the attention mask has a value of `0`. During the self-attention calculation, the dot product between the query and key vectors for these padded positions is set to a very small negative number (effectively negative infinity before softmax). This ensures that the softmax output for these positions becomes zero, preventing the model from attending to the padded tokens and thus not incorporating irrelevant information into the contextual representations of the actual input tokens.

## Exercise 2 - Sentiment analysis pipeline
Objective: Use a pretrained DistilBERT sentiment pipeline to classify a sentence.

Instructions:
1. Import the `pipeline` helper from transformers.
2. Build a pipeline that loads `distilbert-base-uncased-finetuned-sst-2-english`.
3. Pass in a sentence and review the predicted label and score.

Deliverables:
- TODO: Record the sentence you tested.
- TODO: Capture the label plus confidence score and interpret the result.


In [ ]:
from transformers import pipeline

sentiment_pipeline = pipeline(
    task="sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english"
)

sentence = "J'ai vraiment adoré ce film ! C'était fantastique et passionnant."
prediction = sentiment_pipeline(sentence)
prediction

### Livrables de l'Exercice 2

- **Phrase testée :** "J'ai vraiment adoré ce film ! C'était fantastique et passionnant."

- **Résultat de la prédiction :**
  ```json
  [{'label': 'POSITIVE', 'score': 0.9998896718025208}]
  ```
  **Interprétation :** Le modèle a prédit un sentiment `POSITIF` avec un score de confiance très élevé de 0.9998896718025208. Cela indique que le modèle est extrêmement confiant que la phrase exprime une opinion positive.

### Réflexion sur l'Exercice 2

- **Le label prédit correspond-il à votre attente ? Pourquoi ou pourquoi pas ?**
  Oui, le label prédit `POSITIVE` correspond parfaitement à mon attente. La phrase "J'ai vraiment adoré ce film ! C'était fantastique et passionnant." contient des termes clairement positifs et des marqueurs d'enthousiasme ("vraiment adoré", "fantastique", "passionnant"), ce qui indique sans ambiguïté un sentiment positif.

- **Quelle est la confiance du modèle et que vous dit le score ?**
  Le modèle est extrêmement confiant, avec un score de `0.9998896718025208`. Ce score, très proche de 1, indique une quasi-certitude de la part du modèle quant à la classification du sentiment comme `POSITIF`. Un score aussi élevé suggère que la phrase est un exemple très clair et sans ambiguïté du sentiment positif pour ce modèle particulier.

## Exercise 3 - Custom sentiment analyzer class
Objective: Rebuild the pipeline manually so you control tokenization, tensors, and scoring.

Instructions:
1. Import `AutoTokenizer` and `AutoModelForSequenceClassification`.
2. Implement `BERTSentimentAnalyzer` with methods for initialization, preprocessing, and prediction.
3. Test the class with multiple sentences.

Hints:
- Keep a `max_length` attribute so you can reuse it while tokenizing.
- Apply `torch.softmax` to transform logits into probabilities.
- Return both the label and the probability for clarity.


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from typing import Dict

class BERTSentimentAnalyzer:
    def __init__(self, model_name: str = "distilbert-base-uncased-finetuned-sst-2-english", max_length: int = 128):
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForSequenceClassification.from_pretrained(model_name)
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.model.to(self.device)
        self.max_length = max_length

    def preprocess(self, text: str) -> Dict[str, torch.Tensor]:
        encoding = self.tokenizer(
            text,
            add_special_tokens=True,
            padding="max_length",
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt"
        )
        return {
            "input_ids": encoding["input_ids"].to(self.device),
            "attention_mask": encoding["attention_mask"].to(self.device)
        }

    def predict(self, text: str) -> Dict[str, float]:
        inputs = self.preprocess(text)
        with torch.no_grad():
            outputs = self.model(**inputs)

        logits = outputs.logits
        probabilities = torch.softmax(logits, dim=1)[0].cpu().numpy()

        predicted_class_id = torch.argmax(logits, dim=1).item()
        label = self.model.config.id2label[predicted_class_id]
        probability = probabilities[predicted_class_id]

        return {"label": label, "score": float(probability)}


In [ ]:
# TODO: instantiate your analyzer and test several sentences once the class is ready.
# analyzer = BERTSentimentAnalyzer()
# samples = [
#     "TODO: add a clearly positive statement",
#     "TODO: add a clearly negative statement"
# ]
# for text in samples:
#     print(text)
#     print(analyzer.predict(text))


In [ ]:
analyzer = BERTSentimentAnalyzer()
samples = [
    "J'adore ce nouveau livre, c'est incroyable !",
    "Je déteste cette situation, c'est une perte de temps complète.",
    "Ce restaurant est passable, rien d'extraordinaire."
]
for text in samples:
    print(f"Texte: '{text}'")
    print(f"Prédiction: {analyzer.predict(text)}\n")

### Livrables de l'Exercice 3

Voici les prédictions pour les phrases testées avec la classe `BERTSentimentAnalyzer`:

- **Texte:** 'J'adore ce nouveau livre, c'est incroyable !'
  **Prédiction:** `{'label': 'POSITIVE', 'score': 0.9998632669448853}`

- **Texte:** 'Je déteste cette situation, c'est une perte de temps complète.'
  **Prédiction:** `{'label': 'NEGATIVE', 'score': 0.9996843338012695}`

- **Texte:** 'Ce restaurant est passable, rien d'extraordinaire.'
  **Prédiction:** `{'label': 'NEGATIVE', 'score': 0.9934149980545044}`

## Exercise 4 - BERT for Named Entity Recognition
Objective: Build a lightweight class that runs a token-classification model and maps tokens to entity labels.

Instructions:
1. Import `AutoTokenizer` and `AutoModelForTokenClassification`.
2. Implement `BERTNamedEntityRecognizer` with init plus a `recognize` method.
3. Tokenize sample text, run the model, convert the predictions to entity spans, and test with a short paragraph.

Deliverables:
- TODO: Return a list of dictionaries like `{text, entity, start, end}` for each detected entity.
- TODO: Explain how you handled subword tokens that begin with `##`.


In [ ]:
from transformers import AutoTokenizer, AutoModelForTokenClassification
import torch

class BERTNamedEntityRecognizer:
    def __init__(self, model_name: str = "dslim/bert-base-NER"):
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForTokenClassification.from_pretrained(model_name)
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.model.to(self.device)
        self.id_to_label = self.model.config.id2label

    def recognize(self, text: str):
        # Tokenize with offset_mapping to get character spans
        inputs = self.tokenizer(text, return_tensors="pt", truncation=True, padding=True, return_offsets_mapping=True).to(self.device)
        offset_mapping = inputs["offset_mapping"][0].tolist() # List of (start_char, end_char) for each token

        with torch.no_grad():
            outputs = self.model(**inputs)
        predictions = torch.argmax(outputs.logits, dim=2)[0].cpu().numpy()

        tokens = self.tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])

        entities = []
        current_entity_start_offset = -1
        current_entity_end_offset = -1
        current_entity_type = None

        for i, (token_id, prediction_id, offsets) in enumerate(zip(inputs["input_ids"][0], predictions, offset_mapping)):
            label = self.id_to_label[prediction_id]
            start_char, end_char = offsets
            token_text = self.tokenizer.decode(token_id, skip_special_tokens=False) # Get token text from ID

            # Skip special tokens (CLS, SEP) and padding (offset_mapping will be (0,0) for padding)
            if token_text in self.tokenizer.all_special_tokens or (start_char == 0 and end_char == 0 and token_text == '[PAD]'):
                # If an entity was being tracked, finalize it before skipping
                if current_entity_type is not None and current_entity_start_offset != -1:
                    entities.append({
                        "text": text[current_entity_start_offset:current_entity_end_offset],
                        "entity": current_entity_type,
                        "start": current_entity_start_offset,
                        "end": current_entity_end_offset
                    })
                current_entity_start_offset = -1
                current_entity_end_offset = -1
                current_entity_type = None
                continue

            if label.startswith("B-"):
                # If a previous entity was being tracked, finalize it
                if current_entity_type is not None and current_entity_start_offset != -1:
                    entities.append({
                        "text": text[current_entity_start_offset:current_entity_end_offset],
                        "entity": current_entity_type,
                        "start": current_entity_start_offset,
                        "end": current_entity_end_offset
                    })
                # Start a new entity
                current_entity_start_offset = start_char
                current_entity_end_offset = end_char
                current_entity_type = label[2:] # Remove B- prefix
            elif label.startswith("I-") and current_entity_type is not None and label[2:] == current_entity_type:
                # Continue current entity
                current_entity_end_offset = end_char
            else: # O-tag or mismatched I-tag
                # If an entity was being tracked, finalize it
                if current_entity_type is not None and current_entity_start_offset != -1:
                    entities.append({
                        "text": text[current_entity_start_offset:current_entity_end_offset],
                        "entity": current_entity_type,
                        "start": current_entity_start_offset,
                        "end": current_entity_end_offset
                    })
                # Reset entity tracking
                current_entity_start_offset = -1
                current_entity_end_offset = -1
                current_entity_type = None

        # After the loop, if an entity is still open, add it
        if current_entity_type is not None and current_entity_start_offset != -1:
            entities.append({
                "text": text[current_entity_start_offset:current_entity_end_offset],
                "entity": current_entity_type,
                "start": current_entity_start_offset,
                "end": current_entity_end_offset
            })

        return entities

In [ ]:
# TODO: instantiate the recognizer and test it on text that includes people, places, or organizations.
# ner = BERTNamedEntityRecognizer()
# sample_text = "TODO: add a short paragraph with at least two entities."
# ner.recognize(sample_text)


In [ ]:
ner = BERTNamedEntityRecognizer()
sample_text = "Google a été fondé par Larry Page et Sergey Brin en 1998, en Californie. Sundar Pichai est l'actuel PDG de Google et d'Alphabet."
entities = ner.recognize(sample_text)
print(entities)


### Livrables de l'Exercice 4

**1. Entités détectées :**
L'exécution du code ci-dessus retourne une liste de dictionnaires identifiant les organisations (ORG), les personnes (PER) et les lieux (LOC) avec leurs positions exactes dans le texte.

**2. Gestion des sous-mots (`##`) :**
Pour gérer les sous-mots générés par le tokenizer WordPiece de BERT (comme `Alphabet` qui pourrait être découpé en `Alpha` et `##bet`), j'ai utilisé le paramètre `return_offsets_mapping=True` lors de la tokenisation.

Voici comment cela fonctionne :
- Chaque token, qu'il soit un mot complet ou un sous-mot, est associé à un tuple `(start, end)` représentant ses indices de caractères exacts dans la chaîne de texte originale.
- Lorsqu'une entité commence (étiquette `B-`), nous enregistrons l'indice de début.
- Tant que les tokens suivants appartiennent à la même entité (étiquette `I-`), nous mettons à jour l'indice de fin avec la position de fin du dernier token.
- Cela permet de reconstruire le segment de texte original `text[start:end]` sans se soucier des préfixes `##`, car nous extrayons directement la portion correspondante de la phrase source.

## Exercise 5 - Comparing BERT and GPT
Objective: Summarize how encoder-style models differ from decoder-style models.

Fill the table with concise statements (one line each).

| Category | BERT (Encoder-only) | GPT (Decoder-only) |
|----------|------|-----|
| Architecture | Attention bidirectionnelle (contexte gauche et droit) | Attention causale/unidirectionnelle (contexte gauche uniquement) |
| Primary purpose | Compréhension profonde du langage (NLU) | Génération de texte fluide (NLG) |
| Typical use cases | Classification, NER, Questions-Réponses | Chatbots, rédaction, complétion de code |
| Strengths | Capture d'un contexte global et riche pour chaque mot | Capacité impressionnante à produire du texte cohérent |
| Weaknesses | Inefficace pour la génération de texte long | Moins performant pour l'extraction précise d'informations sans exemples |

## Exercise 6 - BERT inside Retrieval-Augmented Generation
Objective: Explain how BERT-generated embeddings power the retrieval stage of a RAG workflow.

1. **Encodage des requêtes et documents :**
BERT transforme des phrases ou des paragraphes entiers en vecteurs numériques de haute dimension (embeddings). Contrairement aux méthodes classiques, ces vecteurs capturent la sémantique profonde : deux phrases avec des mots différents mais un sens similaire (ex: 'Où est la tour Eiffel ?' et 'Localisation du monument parisien') se retrouveront proches dans l'espace vectoriel.

2. **Stockage et recherche vectorielle :**
Ces embeddings sont stockés dans une base de données vectorielle (comme Pinecone ou Chroma). Lorsqu'un utilisateur pose une question, sa requête est également encodée par BERT. Le système effectue ensuite une 'recherche de similarité cosinus' pour trouver instantanément les documents dont les vecteurs sont les plus proches de celui de la requête.

3. **Passage au modèle génératif :**
Une fois les passages les plus pertinents récupérés grâce à BERT, ils sont insérés dans le 'contexte' du prompt envoyé à un modèle comme GPT. Le modèle génératif utilise alors ces informations factuelles pour rédiger une réponse précise, réduisant ainsi les risques d'hallucinations.

4. **Exemple d'application concrète :**
Un assistant technique pour une entreprise aéronautique. BERT permet de retrouver le paragraphe exact dans des milliers de pages de manuels de maintenance complexes à partir d'une description de panne en langage naturel, puis GPT synthétise la procédure de réparation pour le technicien.